X_test = [
  [1.5, 2.5]
]
X_train = [
  [1, 2],
  [2, 3],
  [3, 4],
  [4, 5]
]
y_train = [
  0,
  0,
  1,
  1
]


In [ ]:
import numpy as np

class DecisionTreeClassifier:
    class _Node:
        def __init__(self, is_leaf, prob, feat, thr, left, right):
            self.is_leaf = is_leaf
            self.prob = prob
            self.feat = feat
            self.thr = thr
            self.left = left
            self.right = right

    def __init__(self, max_depth=3, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        self.root = self._build(X, y, 0)

    def predict_proba(self, X):
        return [self._predict_row(row) for row in X]

    def _predict_row(self, row):
        node = self.root
        while not node.is_leaf:
            if row[node.feat] <= node.thr:
                node = node.left
            else:
                node = node.right
        return node.prob

    def _build(self, X, y, depth):
        n = len(y)
        ones = sum(y)
        prob = ones / n
        if ones == 0 or ones == n:
            return self._Node(True, prob, None, None, None, None)
        if depth >= self.max_depth:
            return self._Node(True, prob, None, None, None, None)
        if n < self.min_samples_split:
            return self._Node(True, prob, None, None, None, None)

        best = self._best_split(X, y)
        if best is None:
            return self._Node(True, prob, None, None, None, None)

        feat, thr, left_idx, right_idx = best
        if not left_idx or not right_idx:
            return self._Node(True, prob, None, None, None, None)

        left_X = [X[i] for i in left_idx]
        left_y = [y[i] for i in left_idx]
        right_X = [X[i] for i in right_idx]
        right_y = [y[i] for i in right_idx]

        left_node = self._build(left_X, left_y, depth + 1)
        right_node = self._build(right_X, right_y, depth + 1)

        return self._Node(False, prob, feat, thr, left_node, right_node)

    def _gini(self, y):
        n = len(y)
        if n == 0:
            return 0.0
        ones = sum(y)
        p1 = ones / n
        p0 = 1.0 - p1
        return 1.0 - p0 * p0 - p1 * p1

    def _best_split(self, X, y):
        n_samples = len(y)
        n_features = len(X[0])
        base = self._gini(y)

        best_imp = base
        best_feat = None
        best_thr = None
        best_left = None
        best_right = None

        for f in range(n_features):
            vals = [X[i][f] for i in range(n_samples)]
            uniq = sorted(set(vals))
            if len(uniq) <= 1:
                continue

            thresholds = [
                (uniq[i] + uniq[i + 1]) / 2.0
                for i in range(len(uniq) - 1)
            ]

            for thr in thresholds:
                left_idx = [i for i in range(n_samples)
                            if X[i][f] <= thr]
                right_idx = [i for i in range(n_samples)
                             if X[i][f] > thr]
                if not left_idx or not right_idx:
                    continue

                left_y = [y[i] for i in left_idx]
                right_y = [y[i] for i in right_idx]
                g_left = self._gini(left_y)
                g_right = self._gini(right_y)

                w_left = len(left_idx) / n_samples
                w_right = len(right_idx) / n_samples
                imp = w_left * g_left + w_right * g_right

                if imp < best_imp:
                    best_imp = imp
                    best_feat = f
                    best_thr = thr
                    best_left = left_idx
                    best_right = right_idx
                elif imp == best_imp and best_feat is not None:
                    if f < best_feat or (f == best_feat and thr < best_thr):
                        best_feat = f
                        best_thr = thr
                        best_left = left_idx
                        best_right = right_idx

        if best_feat is None or best_imp >= base:
            return None

        return best_feat, best_thr, best_left, best_right

# Instantiate model
model = DecisionTreeClassifier()
# Fit model
model.fit(X_train, y_train)
# Get predictions
y_test = model.predict_proba(X_test)
# Output predictions
y_test